In [62]:
"""
==========================================================================
Robust Signal-to-Noise Ratio Inference under Volatility Clustering
and Non-Gaussian Heavy Tails: Extensions via Subsampling
and Self-Normalization
 
Author  : Nihar Mahesh Jani
          niharmaheshjani@gmail.com
 
Inspiration & Attribution:
  This work is independently developed and was deeply inspired by
  the ideas in —
 
    "Signal-to-Noise Ratio Inference under Volatility Clustering &
     Heavy Tails" by López de Prado, Porcu, Zoonekynd & Engle (2026),
     available at SSRN: https://ssrn.com/abstract=6568702
 
  This code is NOT the official implementation of that paper, nor is
  it affiliated with or endorsed by any of its authors. Every extension
  here — the HAC variance estimator (E1), the stationary bootstrap (E2),
  the subsampling confidence interval (E3), and the Ibragimov–Müller
  group inference procedure (E4) — was independently conceived, coded,
  and tested by Nihar Mahesh Jani.
 
  If this code helps you understand SNR inference better, I hope you
  will trace the ideas back to López de Prado et al. (2026), who
  asked the important questions first.
 
Overview:
  The classical plug-in Sharpe ratio estimator is widely trusted but its
  confidence intervals are routinely too narrow when returns are clustered
  in volatility or drawn from heavy-tailed distributions. This code
  quantifies exactly how wrong those intervals can be, and then offers
  four increasingly robust alternatives — two of which (subsampling and
  IM group inference) remain valid even when the fourth moment of returns
  does not exist.
 
Methods implemented:
  [Original] Paper CI — closed-form from Proposition 4.7 of the paper.
             Requires finite fourth moment. Best in class when the
             GARCH model is correctly specified.
  [E1] HAC CI — Newey–West long-run variance, delta-method gradient.
             Model-free within the finite-fourth-moment regime.
             Narrower than the paper CI in practice.
  [E2] Stationary Bootstrap — Politis–Romano (JASA 1994) with
             Politis–White (2004) automatic block-length selection.
             Introduced here as a natural first extension; diagnosed
             to stall near 92 % coverage in the stable-law regime.
  [E3] Subsampling — Politis–Romano–Wolf (1999), Chapter 11.
             Does not require a plug-in estimate of the stable index.
             Valid in both the finite-fourth-moment and stable-law
             regimes. Primary recommendation for heavy-tailed data.
  [E4] Ibragimov–Müller Group Inference — JBES (2010).
             Splits the series into q blocks and applies a t-test
             to the block-level SR estimates. Self-normalised: needs
             no knowledge of the stable index or the long-run variance.
             Used here as a robustness benchmark alongside subsampling.
 
Regime separation (the structural insight at the heart of this work):
  FINITE-FOURTH-MOMENT REGIME (DIAG > 0):
    HAC inference is valid and preferred. Subsampling and IM are also
    valid but produce wider intervals.
  STABLE-LAW REGIME (DIAG ≤ 0, Theorem 5.3):
    HAC logic breaks. Paper CI is wrong. Bootstrap stalls.
    Use subsampling (E3) or IM group inference (E4).
 
Mathematical details:
  -----------------------------------------------------------------------
  Subsampling CI (Politis–Romano–Wolf 1999, Chapter 11):
    For τ_n = √n (SR̂ − SR) with unknown limiting distribution:
    (i)  Compute SR̂_n from the full series of length n.
    (ii) For each block of length b, compute SR̂_{n,b,t} on R[t:t+b].
    (iii)Approximate the CDF of √b(SR̂_{n,b,t} − SR̂_n) across all
         n−b+1 blocks.
    (iv) Invert to get:   CI = SR̂_n ± q̂_{1−α/2} · n^{-1/2}
    Block size b = n^{1/3} — standard for heavy-tailed means.
 
  Stationary Bootstrap (Politis–Romano JASA 1994):
    Resamples blocks of geometrically distributed random length from
    the circular data. Block length parameter p = 1/b̄, where b̄ is
    the mean block length chosen by the Politis–White (2004) rule.
 
  Ibragimov–Müller (JBES 2010) Group Inference:
    (i)  Split the series into q non-overlapping contiguous blocks.
    (ii) Compute SR̂_j for j = 1, …, q.
    (iii)μ̂_q = mean(SR̂_j),  s_q = std(SR̂_j, ddof=1).
    (iv) 95 % CI: μ̂_q ± t_{q−1, 0.975} · s_q / √q.
    Valid when (a) block estimators are asymptotically normal or stable,
    and (b) they are asymptotically independent — both approximately
    satisfied for non-overlapping GARCH blocks.
    Caution: in the stable-law regime, infinite-variance block estimates
    make the t-quantile approximate. Still empirically competitive.
  -----------------------------------------------------------------------
 
==========================================================================
"""
 

'\n==========================================================================\nRobust Signal-to-Noise Ratio Inference under Volatility Clustering\nand Non-Gaussian Heavy Tails: Extensions via Subsampling\nand Self-Normalization\n\nAuthor  : Nihar Mahesh Jani\n          niharmaheshjani@gmail.com\n\nInspiration & Attribution:\n  This work is independently developed and was deeply inspired by\n  the ideas in —\n\n    "Signal-to-Noise Ratio Inference under Volatility Clustering &\n     Heavy Tails" by López de Prado, Porcu, Zoonekynd & Engle (2026),\n     available at SSRN: https://ssrn.com/abstract=6568702\n\n  This code is NOT the official implementation of that paper, nor is\n  it affiliated with or endorsed by any of its authors. Every extension\n  here — the HAC variance estimator (E1), the stationary bootstrap (E2),\n  the subsampling confidence interval (E3), and the Ibragimov–Müller\n  group inference procedure (E4) — was independently conceived, coded,\n  and tested by Nihar Mahes

# Install Library 

In [63]:
! pip install numpy scipy pandas matplotlib numba yfinance arch

# Import Libraries 

In [64]:
import warnings
warnings.filterwarnings("ignore")

In [65]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from numba import njit
import time

In [66]:
from arch import arch_model
import yfinance as yf

In [67]:
# Setting the seed upfront means every table and figure in this
# paper can be reproduced exactly from this single script.
np.random.seed(42)

# SECTION 1 — PARAMETERS

In [68]:
# Calibrated to the MSCI World daily return series, following the
# parameter choices in López de Prado et al. (2026), Table 1.
# I have kept these values unchanged so that my simulation results
# are directly comparable to those in the paper.

In [69]:
MU    = 5e-4          # Unconditional daily mean return
ALPHA = 0.13          # GARCH(1,1) shock coefficient
BETA  = 0.81          # GARCH(1,1) persistence coefficient
PHI   = ALPHA + BETA  # Total persistence = 0.94 — strongly clustered
S2    = 9e-4          # Unconditional daily variance (σ ≈ 3 % per day)
KZ    = 6.0           # Fourth standardised moment of innovations (Student-t, df=6)
OMEGA = S2 * (1.0 - PHI)
TRUE_SR = MU / np.sqrt(S2)   # Population SR ≈ 0.01667

In [70]:
# Fourth-moment diagnostic from Proposition 4.7 of the paper.
# When DIAG > 0 the asymptotic variance V_sym is finite and the
# paper's closed-form CI applies. When DIAG ≤ 0 we are in the
# stable-law regime and must use E3 or E4.
DIAG = 1.0 - ALPHA**2 * KZ - 2.0 * ALPHA * BETA - BETA**2
assert DIAG > 0, f"Fourth-moment condition violated: DIAG={DIAG:.5f}"

In [71]:
# Paper Proposition 4.7 — closed-form asymptotic variance V_sym.
# Under conditional symmetry the skewness channel Ω_yu vanishes.
V_PAPER = (
    1.0
    + (MU**2 / (4.0 * S2))
    * ((1.0 - BETA)**2 * (KZ - 1.0) * (1.0 + PHI))
    / ((1.0 - PHI) * DIAG)
)
V_IID = 1.0 + TRUE_SR**2 / 2.0   # i.i.d. Gaussian benchmark for comparison

# SECTION 2 — DATA GENERATION

In [72]:
# Three DGPs cover the two regimes identified in the paper:
#   (a) GARCH(1,1) with Student-t innovations — correct specification
#   (b) EGARCH(1,1) — misspecified but still finite-fourth-moment
#   (c) Stable-law — Theorem 5.3 regime, infinite fourth moment
# Numba JIT compilation brings the inner loops to near-C speed,
# making 400-repetition Monte Carlo feasible in a few minutes.

In [73]:
# Module-level constants passed into JIT kernels by value.
_OMEGA  = float(OMEGA)
_ALPHA  = float(ALPHA)
_BETA   = float(BETA)
_MU     = float(MU)
_S2     = float(S2)

In [74]:
@njit(fastmath=True)
def _garch11_kernel(z_arr: np.ndarray, burn: int) -> np.ndarray:
    """
    JIT-compiled GARCH(1,1) loop.
 
    Implements Equation (8) of López de Prado et al. (2026):
        y_t = σ_t z_t,   σ²_t = ω + α y²_{t-1} + β σ²_{t-1}
 
    z_arr   : pre-drawn, pre-clipped standardised innovations
    burn    : number of burn-in steps to discard
    Returns : array of excess returns R[burn:] of length T
    """
    N = len(z_arr)
    T = N - burn
    s2_t = _S2
    y_prev = 0.0
    for t in range(1, burn):
        y_t   = (s2_t ** 0.5) * z_arr[t]
        s2_t  = _OMEGA + _ALPHA * y_prev * y_prev + _BETA * s2_t
        if s2_t < 1e-12: s2_t = 1e-12
        if s2_t > 0.5:   s2_t = 0.5
        y_prev = y_t
    out = np.empty(T)
    for i in range(T):
        t = burn + i
        y_t   = (s2_t ** 0.5) * z_arr[t]
        out[i] = _MU + y_t
        s2_t  = _OMEGA + _ALPHA * y_prev * y_prev + _BETA * s2_t
        if s2_t < 1e-12: s2_t = 1e-12
        if s2_t > 0.5:   s2_t = 0.5
        y_prev = y_t
    return out

In [75]:
def simulate_garch11(T: int, df_t: float = 6.0,
                     seed: int = None, burn: int = 300) -> np.ndarray:
    """
    GARCH(1,1) returns with Student-t(df_t) innovations.
 
    This is the benchmark DGP — the world in which the paper's
    closed-form CI was designed to operate. Matching the paper's
    calibration (df=6, α=0.13, β=0.81) lets us verify that my
    reproduced coverage numbers align with theirs before moving
    to the extensions.
    """
    df_t = max(float(df_t), 2.02)
    if seed is not None:
        np.random.seed(seed)
    scale = np.sqrt(df_t / (df_t - 2.0))
    N = T + burn
    z_raw = np.random.standard_t(df_t, size=N) / scale
    np.clip(z_raw, -7.0, 7.0, out=z_raw)
    return _garch11_kernel(z_raw, burn)

In [76]:
@njit(fastmath=True)
def _egarch_kernel(z_arr: np.ndarray, burn: int) -> np.ndarray:
    N = len(z_arr)
    T = N - burn
    log_h = np.log(_S2)
    y_prev = 0.0
    out = np.empty(T)
    for t in range(1, N):
        prev_s = np.exp(0.5 * log_h)
        if prev_s < 1e-8:
            prev_s = 1e-8
        zp = y_prev / prev_s
        if zp >  5.0: zp =  5.0
        if zp < -5.0: zp = -5.0
        log_h = 0.96 * log_h + 0.21 * (abs(zp) - 0.7979) - 0.04 * zp
        if log_h < -12.0: log_h = -12.0
        if log_h >   4.0: log_h =   4.0
        y_t = np.exp(0.5 * log_h) * z_arr[t]
        if t >= burn:
            out[t - burn] = _MU + y_t
        y_prev = y_t
    return out

In [77]:
def simulate_egarch(T: int, seed: int = None, burn: int = 300) -> np.ndarray:
    """
    EGARCH(1,1) — Nelson (1991). Intentionally misspecified DGP.
 
    Running the paper's CI on EGARCH data reveals what happens when
    the practitioner uses the right inferential idea but the wrong
    model. The HAC extension (E1) partially recovers here because
    it is model-free within the finite-fourth-moment regime.
 
    Calibrated so the unconditional variance ≈ S2, keeping the
    spread comparable to the GARCH DGP.
    """
    if seed is not None:
        np.random.seed(seed)
    N = T + burn
    z_arr = np.random.standard_normal(N)
    return _egarch_kernel(z_arr, burn)

In [78]:
def simulate_stable_law(T: int, alpha_stable: float = 1.7,
                        seed: int = None, burn: int = 300) -> np.ndarray:
    """
    Stable-law DGP — corresponds to Theorem 5.3 of López de Prado et al.
    (2026). Added by Nihar Mahesh Jani as the empirical test-bed for E3
    and E4.
 
    Tail index α_stable ∈ (1, 2) implies infinite fourth moment, so
    the fourth-moment diagnostic DIAG would be ≤ 0 if we tried to
    apply the GARCH formula. Standard HAC and bootstrap CIs break here.
 
    Generation uses the Chambers–Mallows–Stuck parametrisation of a
    symmetric stable distribution S(α_s, 0, σ, μ). Because strict
    variance does not exist for α_s < 2, the spread is matched to the
    GARCH DGP via the interquartile range rather than the variance.
 
    This DGP is the stress test: only subsampling (E3) and IM group
    inference (E4) maintain reliable coverage here.
    """
    if seed is not None:
        np.random.seed(seed)
    N = T + burn
    U = np.random.uniform(-np.pi/2, np.pi/2, N)
    E = np.random.exponential(1.0, N)
    if abs(alpha_stable - 1.0) < 1e-6:
        z = (2.0/np.pi) * ((np.pi/2 + 0.0*U)*np.tan(U)
            - np.log((np.pi/2*E*np.cos(U))/(np.pi/2 + 0.0*U)))
    else:
        B = 0.0
        z = (np.sin(alpha_stable * (U + B * np.pi/(2*alpha_stable)))
             / (np.cos(U)**(1.0/alpha_stable))
             * (np.cos(U - alpha_stable*(U + B*np.pi/(2*alpha_stable)))/E)
             ** ((1.0 - alpha_stable)/alpha_stable))
    iqr_z = np.percentile(z, 75) - np.percentile(z, 25)
    target_iqr = 2.0 * np.sqrt(S2) * 1.35
    z = z * (target_iqr / max(iqr_z, 1e-8))
    R = MU + z
    R = np.clip(R, -0.5, 0.5)
    return R[burn:].copy()

# SECTION 3 — PAPER REPRODUCTION

In [79]:
# I have reproduced the paper's core results exactly as written.
# Nothing here is my own contribution; credit belongs entirely to
# López de Prado, Porcu, Zoonekynd & Engle (2026).

In [80]:
def plug_in_sharpe(R: np.ndarray) -> float:
    """
    Plug-in Sharpe ratio estimator — Equation (2) of the paper.
    SR̂ = R̄ / σ̂,  where σ̂ uses ddof=1.
    """
    sigma = np.std(R, ddof=1)
    return np.mean(R) / max(sigma, 1e-12)

In [81]:
def paper_closed_form_V(alpha: float = ALPHA, beta: float = BETA,
                         kz: float = KZ, mu: float = MU,
                         s2: float = S2) -> float:
    """
    Proposition 4.7 — symmetric GARCH(1,1) asymptotic variance V_sym.
 
    Under conditional symmetry the skewness channel Ω_yu = 0, giving:
      V_sym = 1 + (μ²/4σ²) · (1−β)²(κ_z−1)(1+φ) / [(1−φ)·DIAG]
 
    Returns NaN when the fourth-moment condition is violated so that
    downstream code can detect the regime automatically.
    """
    phi  = alpha + beta
    diag = 1.0 - alpha**2 * kz - 2.0 * alpha * beta - beta**2
    if diag <= 0:
        return np.nan
    return (1.0
            + (mu**2 / (4.0 * s2))
            * ((1.0 - beta)**2 * (kz - 1.0) * (1.0 + phi))
            / ((1.0 - phi) * diag))

In [82]:
def paper_ci(sr_hat: float, T: int, alpha_level: float = 0.05):
    """
    Paper CI — fixed width based on the closed-form V from Prop 4.7.
    Half-width = z_{α/2} · √(V_paper / T).
    Valid in the finite-fourth-moment regime only.
    """
    z  = stats.norm.ppf(1.0 - alpha_level / 2.0)
    se = np.sqrt(V_PAPER / T)
    return sr_hat - z * se, sr_hat + z * se

# SECTION 4 — EXTENSION E1: HAC VARIANCE ESTIMATOR

In [83]:
# SECTION 4 — EXTENSION E1: HAC VARIANCE ESTIMATOR
# Original contribution by Nihar Mahesh Jani.
#
# Motivation: the paper's CI uses a model-specific closed form that
# requires knowing α, β, and κ_z. In practice these are estimated
# with error. A HAC-based delta-method variance is model-free and
# adapts to any weakly-dependent DGP — it only asks that the fourth
# moment is finite. In the GARCH-correct case it is narrower than the
# paper CI, which is an encouraging sign that the closed form is
# somewhat conservative.

In [84]:
def _newey_west_lrcov(a: np.ndarray, b: np.ndarray,
                      max_lag: int = None) -> float:
    """
    Newey–West (1987) Bartlett-kernel long-run covariance estimator.
 
    Ω̂(a,b) = Σ_{h=−M}^{M} (1 − |h|/(M+1)) γ̂_{ab}(h)
 
    Bandwidth M is chosen by the Andrews (1991) AR(1) plug-in rule.
    The inner loop is vectorised over lags using numpy dot products,
    which cuts runtime by roughly 10× compared to a Python for-loop.
    """
    n = len(a)
    if max_lag is None:
        max_lag = max(int(4.0 * (n / 100.0) ** (2.0 / 9.0)), 3)
    ac = a - a.mean()
    bc = b - b.mean()
    cov = np.mean(ac * bc)
    lags = np.arange(1, min(max_lag + 1, n))
    weights = 1.0 - lags / (max_lag + 1.0)
    for h, w in zip(lags, weights):
        cov += 2.0 * w * np.dot(ac[h:], bc[:-h]) / (n - h)
    return float(cov)

In [85]:
def hac_asymptotic_V(R: np.ndarray) -> float:
    """
    E1: HAC estimate of the asymptotic variance V of SR̂.
 
    Delta-method gradients for SR = μ / σ:
      g₁ = ∂SR/∂μ = 1/σ  ∝  m₂/σ³
      g₂ = ∂SR/∂m₂ = −μ/(2σ³)
 
    Long-run covariance matrix of (R, R²) estimated via Newey–West.
    Resulting variance:
      V̂_HAC = g₁² Ω̂₁₁ + 2g₁g₂ Ω̂₁₂ + g₂² Ω̂₂₂
 
    This breaks in the stable-law regime because Ω̂₁₁ is then
    estimating the long-run variance of a process with no variance —
    exactly the regime-mismatch failure described in the paper.
    """
    mu  = np.mean(R)
    m2  = np.mean(R ** 2)
    u1  = R - mu
    u2  = R**2 - m2
    O11 = _newey_west_lrcov(u1, u1)
    O12 = _newey_west_lrcov(u1, u2)
    O22 = _newey_west_lrcov(u2, u2)
    s2  = np.var(R, ddof=1)
    s3  = s2 ** 1.5
    g1  = m2 / s3
    g2  = -mu / (2.0 * s3)
    V = g1**2 * O11 + 2.0 * g1 * g2 * O12 + g2**2 * O22
    return max(V, 1e-12)

In [86]:
def hac_ci(R: np.ndarray, alpha_level: float = 0.05):
    """
    E1: 95 % CI for SR using the HAC-estimated asymptotic variance.
    Model-free within the finite-fourth-moment regime.
    Preferred over the paper CI when model parameters are uncertain.
    """
    n    = len(R)
    sr   = plug_in_sharpe(R)
    V    = hac_asymptotic_V(R)
    z    = stats.norm.ppf(1.0 - alpha_level / 2.0)
    se   = np.sqrt(V / n)
    return sr - z * se, sr + z * se

# SECTION 5 — EXTENSION E2: STATIONARY BOOTSTRAP CI

In [87]:
# Original contribution by Nihar Mahesh Jani.
#
# The stationary bootstrap is the natural first step beyond HAC: it
# makes no distributional assumption and handles serial dependence by
# resampling overlapping circular blocks of geometrically random length.
# However, as my simulation results and subsequent discussion with
# Enrique Porcu confirmed, the bootstrap implicitly assumes finite
# variance, causing coverage to plateau around 92 % in the stable-law
# regime rather than converging to 95 %. This stalling is documented
# in detail in Section 9 and motivates the move to E3 and E4.

In [88]:
def _politis_white_block_length(R: np.ndarray) -> int:
    """
    Politis–White (2004) / Patton–Politis–White (2009) automatic
    block-length selection for the stationary bootstrap.
 
    The optimal mean block length b̄* balances bias and variance of
    the bootstrap distribution estimate:
      b̄* = [2G²/D]^{1/3} · n^{1/3}
    where G = Σ_{k=1}^M k γ(k) and D = [2 Σ_{k=1}^M γ(k)]².
    γ(k) is the sample autocovariance of R at lag k, and M is
    a flat-top lag-window bandwidth chosen as ⌊√n⌋.
 
    The resulting geometric parameter is p* = 1/b̄*, clipped to
    the range [2, n/3] for stability.
    """
    n  = len(R)
    x  = R - R.mean()
    M  = max(int(np.floor(np.sqrt(n))), 5)
    M  = min(M, n // 4)
    acf = np.array([np.mean(x[k:] * x[:-k]) if k > 0 else np.mean(x**2)
                    for k in range(M + 1)])
    G   = sum(k * acf[k] for k in range(1, M + 1))
    D   = (2.0 * sum(acf[k] for k in range(1, M + 1))) ** 2
    if D < 1e-15 or G < 0:
        return max(int(n**0.25), 2)
    b_opt = (2.0 * G**2 / max(D, 1e-15)) ** (1.0/3.0) * n**(1.0/3.0)
    b_opt = int(np.clip(np.round(b_opt), 2, n // 3))
    return b_opt
 

In [89]:
@njit(fastmath=True)
def _stationary_bootstrap_stats(R_circ: np.ndarray, n: int, B: int,
                                  p: float, sr_hat: float,
                                  seeds_start: np.ndarray,
                                  lens_arr: np.ndarray) -> np.ndarray:
    """
    JIT-compiled core of the stationary bootstrap.
 
    All random draws (block starts and block lengths) are generated
    outside this function and passed in as arrays, so the Numba loop
    only assembles pseudo-series and computes the SR statistic —
    the two operations that genuinely benefit from compiled speed.
 
    R_circ      : circular wrap of R (length 2n), enabling wrap-around
    seeds_start : (B, max_blocks) array of block start positions
    lens_arr    : (B, max_blocks) array of geometric block lengths
    Returns     : array of bootstrap statistics √n(SR̂*_b − SR̂_n)
    """
    max_blocks = seeds_start.shape[1]
    boot_stats = np.empty(B)
    sqrt_n = n ** 0.5
    for b in range(B):
        boot = np.empty(n)
        idx = 0
        block_i = 0
        while idx < n and block_i < max_blocks:
            blen = lens_arr[b, block_i]
            if blen > n - idx:
                blen = n - idx
            start = seeds_start[b, block_i]
            for k in range(blen):
                boot[idx + k] = R_circ[start + k]
            idx += blen
            block_i += 1
        mu_b = 0.0
        for i in range(n):
            mu_b += boot[i]
        mu_b /= n
        s2_b = 0.0
        for i in range(n):
            d = boot[i] - mu_b
            s2_b += d * d
        s2_b /= (n - 1)
        sr_b = mu_b / (s2_b ** 0.5 + 1e-12)
        boot_stats[b] = sqrt_n * (sr_b - sr_hat)
    return boot_stats

In [90]:
def stationary_bootstrap_ci(R: np.ndarray, B: int = 999,
                             alpha_level: float = 0.05,
                             seed: int = None) -> tuple:
    """
    E2: Stationary Bootstrap CI for SR — Politis & Romano (JASA 1994).
 
    The pivot CI inverts the empirical quantiles of √n(SR̂*_b − SR̂_n):
      lo = SR̂_n − q̂_{1−α/2} / √n
      hi = SR̂_n − q̂_{α/2}   / √n
 
    Block-length parameter is selected automatically via Politis–White
    (2004). All B bootstrap samples are assembled inside the JIT kernel
    from pre-generated geometric block lengths and start indices, which
    removes the Python-loop bottleneck entirely.
 
    Limitation (identified in my simulations, confirmed by Porcu):
    In the stable-law regime the bootstrap implicitly assumes finite
    variance, so coverage stalls near 92 % and does not converge to
    95 % as T grows. E3 (subsampling) resolves this.
    """
    if seed is not None:
        np.random.seed(seed)
    n     = len(R)
    sr_hat = plug_in_sharpe(R)
    b_bar  = _politis_white_block_length(R)
    p      = 1.0 / max(b_bar, 1)
    R_circ = np.concatenate([R, R])
    max_blocks = int(np.ceil(n / b_bar) * 6) + 20
    rng = np.random.default_rng(seed)
    raw_geom = rng.geometric(p, size=(B, max_blocks)).astype(np.int32)
    np.clip(raw_geom, 1, n, out=raw_geom)
    starts = rng.integers(0, n, size=(B, max_blocks)).astype(np.int32)
    boot_stats = _stationary_bootstrap_stats(R_circ, n, B, p,
                                              sr_hat, starts, raw_geom)
    q_lo = np.percentile(boot_stats, 100 * alpha_level / 2.0)
    q_hi = np.percentile(boot_stats, 100 * (1.0 - alpha_level / 2.0))
    lo = sr_hat - q_hi / np.sqrt(n)
    hi = sr_hat - q_lo / np.sqrt(n)
    return lo, hi

# SECTION 6 — EXTENSION E3: SUBSAMPLING CI

In [91]:
# Original contribution by Nihar Mahesh Jani.
#
# After observing the stationary bootstrap stall in the stable-law
# regime, I looked for an inference method that makes no assumption
# about the existence of moments. Subsampling (Politis–Romano–Wolf
# 1999) fits that description perfectly: it approximates the sampling
# distribution of SR̂ from the data itself, with no requirement on the
# tail index. This is the primary method I recommend for practitioners
# who face heavy-tailed return distributions.

In [92]:
def subsampling_ci(R: np.ndarray, alpha_level: float = 0.05) -> tuple:
    """
    E3: Subsampling CI for SR — Politis, Romano & Wolf (1999), Ch. 11.
 
    Key idea: the distribution of √b(SR̂_{n,b,t} − SR̂_n) over all
    n−b+1 overlapping subsamples of length b approximates the
    distribution of √n(SR̂_n − SR) at rate (b/n)^{1/2}, without
    requiring any knowledge of the tail index or long-run variance.
 
    CI (Theorem 2.2.1 of PRW 1999):
      lo = SR̂_n − q̂_{1−α/2} / √n
      hi = SR̂_n − q̂_{α/2}   / √n
 
    Block size: b = ⌊n^{1/3}⌋ — standard for heavy-tailed means
    (PRW 1999, Chapter 11). Larger b introduces bias; smaller b
    inflates variance. The cube-root rate is the right trade-off.
 
    Implementation uses stride tricks to build the full
    (n_sub × b) submatrix as a zero-copy view, then computes
    all subsample means and variances in one vectorised pass.
 
    Valid in both the finite-fourth-moment regime and the stable-law
    regime. Produces slightly wider intervals than HAC in the GARCH
    case — a worthwhile cost for the regime-agnostic guarantee.
    """
    n     = len(R)
    sr_hat = plug_in_sharpe(R)
    b = max(int(np.floor(n**(1.0/3.0))), 5)
 
    from numpy.lib.stride_tricks import as_strided
    n_sub = n - b + 1
    itemsize = R.strides[0]
    sub_matrix = as_strided(R,
                            shape=(n_sub, b),
                            strides=(itemsize, itemsize))
    sub_means = sub_matrix.mean(axis=1)
    sub_vars  = ((sub_matrix - sub_means[:, None])**2).sum(axis=1) / (b - 1)
    sub_srs   = sub_means / np.maximum(np.sqrt(sub_vars), 1e-12)
 
    sub_stats = np.sqrt(b) * (sub_srs - sr_hat)
 
    q_lo = np.percentile(sub_stats, 100 * alpha_level / 2.0)
    q_hi = np.percentile(sub_stats, 100 * (1.0 - alpha_level / 2.0))
 
    lo = sr_hat - q_hi / np.sqrt(n)
    hi = sr_hat - q_lo / np.sqrt(n)
    return lo, hi

# SECTION 7 — EXTENSION E4: IBRAGIMOV–MÜLLER GROUP INFERENCE

In [93]:
# Original contribution by Nihar Mahesh Jani.
#
# Subsampling is the primary recommendation; IM group inference serves
# as a robustness benchmark that requires essentially no tuning.
# Splitting the series into q blocks and running a t-test on the block
# SR estimates is an elegant idea: the t-test self-normalises, so the
# practitioner needs neither a stable-index estimate nor a bandwidth
# choice. The only tuning parameter is q, and the results are
# reassuringly stable across q ∈ {4, 8, 16}.

In [94]:
def ibragimov_muller_ci(R: np.ndarray, q: int = 8,
                         alpha_level: float = 0.05) -> tuple:
    """
    E4: Ibragimov–Müller (JBES 2010) group t-test CI for SR.
 
    Algorithm:
      1. Partition R into q contiguous, non-overlapping blocks.
      2. Compute SR̂_j for each block j = 1, …, q.
      3. Apply a one-sample t-CI with df = q − 1:
           μ̂_q ± t_{q−1, 1−α/2} · (s_q / √q)
 
    Why this works (Theorem 1, IM 2010):
      - Each SR̂_j is asymptotically normal (finite-variance regime)
        or stable (Theorem 5.3 regime) as the block length grows.
      - Non-overlapping blocks from an α-mixing process are
        asymptotically independent, satisfying the second condition.
    The t-test critical value absorbs the unknown long-run variance —
    that is what "self-normalised" means in practice.
 
    Caution: in the stable-law regime the block SR̂_j have infinite
    variance, making the t-quantile approximate rather than exact.
    Empirically, however, this method still achieves better coverage
    than the stationary bootstrap in that regime, which is why it
    serves as a useful robustness benchmark.
 
    Default q = 8 balances between:
      - q = 4: t_{3, 0.975} = 3.18, intervals uncomfortably wide
      - q = 16: block length too short for asymptotic independence
    """
    n      = len(R)
    q      = min(q, n // 10)
    q      = max(q, 2)
 
    block_size = n // q
    sr_vals = np.array([
        plug_in_sharpe(R[j * block_size: (j + 1) * block_size])
        for j in range(q)
        if len(R[j * block_size: (j + 1) * block_size]) >= 10
    ])
 
    q_actual = len(sr_vals)
    if q_actual < 2:
        return np.nan, np.nan
 
    mu_q  = np.mean(sr_vals)
    s_q   = np.std(sr_vals, ddof=1)
    t_crit = stats.t.ppf(1.0 - alpha_level / 2.0, df=q_actual - 1)
    se_q  = s_q / np.sqrt(q_actual)
 
    return mu_q - t_crit * se_q, mu_q + t_crit * se_q

# SECTION 8 — COMPREHENSIVE EXPERIMENT: ALL 5 METHODS × 3 DGPs

In [95]:
# This is the central Monte Carlo study. 400 replications at T = 800
# is enough to estimate coverage to ± 1 percentage point with 95 %
# confidence. Each replication evaluates all five CI methods so that
# results are directly comparable across methods within a DGP.
#
# To keep the experiment runtime practical, the paper CI half-width
# is cached by T (it does not depend on the data sample) and the
# JIT-compiled kernels handle the inner loops.

In [96]:
_Z95         = stats.norm.ppf(0.975)
_PAPER_SE_T  = {}   # Cache paper SE by T to avoid repeated sqrt

In [97]:
def _paper_se(T: int) -> float:
    """Cached paper CI standard error — √(V_PAPER / T)."""
    if T not in _PAPER_SE_T:
        _PAPER_SE_T[T] = np.sqrt(V_PAPER / T)
    return _PAPER_SE_T[T]

In [98]:
def run_full_experiment(T: int = 800, n_reps: int = 400) -> pd.DataFrame:
    """
    Monte Carlo coverage and width study — all 5 methods × 3 DGPs.
 
    DGPs:
      GARCH (correct)       — the paper's own setting
      EGARCH (misspecified) — same regime, wrong model
      Stable-law (T5.3)     — Theorem 5.3 of López de Prado et al. (2026)
 
    For each DGP × method combination, the table reports:
      Cover%   : empirical coverage at nominal 95 %
      Width    : mean CI width (narrower is better, given coverage ≥ 95 %)
      W_ratio  : width relative to the Paper CI — < 1 means more efficient
 
    Designed so that a reader can see at a glance which methods
    remain trustworthy when the distributional assumptions break.
    """
    rng_boot = 0
 
    dgps = [
        ("GARCH (correct)",    lambda s: simulate_garch11(T, df_t=6.0, seed=s)),
        ("EGARCH (mispecified)", lambda s: simulate_egarch(T, seed=s)),
        ("Stable-law (T5.3)",   lambda s: simulate_stable_law(T, alpha_stable=1.7, seed=s)),
    ]
 
    se_paper = _paper_se(T)
 
    rows = []
    for label, generator in dgps:
        cover = {"Paper": 0, "HAC": 0, "SB": 0, "Sub": 0, "IM": 0}
        widths = {"Paper": [], "HAC": [], "SB": [], "Sub": [], "IM": []}
 
        for rep in range(n_reps):
            R = generator(rep * 7 + 1)
            R = R[np.isfinite(R)]
            if len(R) < 100:
                continue
 
            sr_hat = plug_in_sharpe(R)
 
            # [1] Paper CI — constant width, computed from cached SE
            lo_p = sr_hat - _Z95 * se_paper
            hi_p = sr_hat + _Z95 * se_paper
            # [2] HAC CI (E1)
            lo_h, hi_h = hac_ci(R)
            # [3] Stationary Bootstrap (E2)
            lo_sb, hi_sb = stationary_bootstrap_ci(R, B=999, seed=rng_boot + rep)
            # [4] Subsampling (E3)
            lo_su, hi_su = subsampling_ci(R)
            # [5] Ibragimov–Müller (E4)
            lo_im, hi_im = ibragimov_muller_ci(R, q=8)
 
            def check_cover(lo, hi):
                if np.isfinite(lo) and np.isfinite(hi):
                    return lo <= TRUE_SR <= hi
                return False
 
            if check_cover(lo_p,  hi_p):  cover["Paper"] += 1
            if check_cover(lo_h,  hi_h):  cover["HAC"]   += 1
            if check_cover(lo_sb, hi_sb): cover["SB"]    += 1
            if check_cover(lo_su, hi_su): cover["Sub"]   += 1
            if check_cover(lo_im, hi_im): cover["IM"]    += 1
 
            if np.isfinite(hi_p - lo_p):   widths["Paper"].append(hi_p - lo_p)
            if np.isfinite(hi_h - lo_h):   widths["HAC"].append(hi_h - lo_h)
            if np.isfinite(hi_sb - lo_sb): widths["SB"].append(hi_sb - lo_sb)
            if np.isfinite(hi_su - lo_su): widths["Sub"].append(hi_su - lo_su)
            if np.isfinite(hi_im - lo_im): widths["IM"].append(hi_im - lo_im)
 
        wp = np.mean(widths["Paper"]) if widths["Paper"] else np.nan
 
        for key in ["Paper", "HAC", "SB", "Sub", "IM"]:
            wk = np.mean(widths[key]) if widths[key] else np.nan
            rows.append({
                "DGP":        label,
                "Method":     key,
                "Cover%":     round(100.0 * cover[key] / n_reps, 1),
                "Width":      round(wk, 5) if np.isfinite(wk) else np.nan,
                "W_ratio":    round(wk / wp, 4) if (np.isfinite(wk) and np.isfinite(wp)) else np.nan,
            })
 
    return pd.DataFrame(rows)

# SECTION 9 — STABLE-LAW REGIME: COVERAGE VERSUS SAMPLE SIZE

In [99]:
# This experiment directly documents the stationary bootstrap's
# failure mode in the stable-law regime. The stalling near 92 %
# at T = 500 is not a finite-sample artefact that disappears with
# more data — it persists because the bootstrap variance estimator
# is conceptually wrong in this regime, not just noisy.
# Subsampling and IM group inference do not exhibit this behaviour.

In [100]:
def stable_regime_coverage_vs_T(T_values=None, n_reps: int = 300) -> pd.DataFrame:
    """
    Coverage versus sample size in the stable-law regime.
 
    Tracks four methods — HAC, Stationary Bootstrap, Subsampling,
    and IM Group — across increasing T. The experiment makes visible
    a pattern that summary tables at a single T would hide: the
    bootstrap plateaus while subsampling and IM converge toward 95 %.
 
    n_reps = 1000 in the main run gives Monte Carlo standard errors
    of ± 0.7 pp, tight enough to distinguish a 92 % plateau from
    a genuinely converging sequence.
    """
    if T_values is None:
        T_values = [200, 350, 500, 750, 1000]
    rows = []
    for T in T_values:
        cover = {"SB": 0, "Sub": 0, "IM": 0, "HAC": 0}
        for rep in range(n_reps):
            R = simulate_stable_law(T, alpha_stable=1.7, seed=rep * 13 + 3)
            R = R[np.isfinite(R)]
            if len(R) < 50:
                continue
            sr_hat = plug_in_sharpe(R)
            lo, hi = stationary_bootstrap_ci(R, B=999, seed=rep)
            if np.isfinite(lo) and lo <= TRUE_SR <= hi:
                cover["SB"] += 1
            lo, hi = subsampling_ci(R)
            if np.isfinite(lo) and lo <= TRUE_SR <= hi:
                cover["Sub"] += 1
            lo, hi = ibragimov_muller_ci(R, q=8)
            if np.isfinite(lo) and lo <= TRUE_SR <= hi:
                cover["IM"] += 1
            lo, hi = hac_ci(R)
            if np.isfinite(lo) and lo <= TRUE_SR <= hi:
                cover["HAC"] += 1
 
        for key in ["HAC", "SB", "Sub", "IM"]:
            rows.append({
                "T": T,
                "Method": key,
                "Cover%": round(100.0 * cover[key] / n_reps, 1),
            })
    return pd.DataFrame(rows)

# SECTION 10 — V_sym CURVE (REPRODUCED FROM PAPER)

In [101]:
# Plots the closed-form asymptotic variance against the persistence
# parameter φ = α + β. The curve diverges as φ → φ_max (the
# fourth-moment boundary), making it a useful visual reminder that
# standard errors can be severely underestimated near that boundary.

In [102]:
def v_sym_curve():
    """
    V_sym versus φ across the feasible persistence range.
 
    Returns the grid and values so the plotting section can overlay
    the calibrated φ = 0.94 against the theoretical curve.
    NaN values (beyond the fourth-moment boundary) are masked in the
    figure so the curve ends cleanly rather than diverging to ±∞.
    """
    phi_grid = np.linspace(0.50, 0.97, 200)
    V_vals   = []
    for ph in phi_grid:
        a = ph * 0.15
        b = ph * 0.85
        V_vals.append(paper_closed_form_V(a, b, KZ, MU, S2))
    return phi_grid, np.array(V_vals, dtype=float)

# SECTION 11 — PLOTTING

In [103]:
# Nine panels arranged in three rows. Row 1 reproduces the two
# core figures from the paper (coverage, width, V_sym curve).
# Rows 2 and 3 show the new results contributed by Nihar Mahesh Jani:
# all five methods across all three DGPs, the coverage-versus-T
# trajectory in the stable-law regime, and a summary of the regime
# separation logic that underlies the extension design.

In [104]:
def make_extended_figure(df_full: pd.DataFrame,
                          df_stable: pd.DataFrame,
                          phi_grid, V_curve,
                          out_path: str = "snr_extended.png"):
    """
    Extended 9-panel figure for the paper
    "Robust SNR Inference under Volatility Clustering and
    Non-Gaussian Heavy Tails: Extensions via Subsampling and
    Self-Normalization" by Nihar Mahesh Jani.
 
    Layout:
      Row 1 (Paper reproduction, preserved):
        [1] Coverage — Paper CI vs HAC, GARCH and EGARCH
        [2] CI width — all five methods on the GARCH DGP
        [3] V_sym versus φ (Proposition 4.7)
      Row 2 (New — all five methods across all three DGPs):
        [4] GARCH regime
        [5] EGARCH regime
        [6] Stable-law regime
      Row 3 (New — stable-law deep dive):
        [7] Coverage versus T for all methods
        [8] Width ratios across DGPs
        [9] Regime separation summary (text panel)
    """
    BG    = "#161b22"
    RED   = "#FF6B6B"
    TEAL  = "#4ECDC4"
    GOLD  = "#FFD700"
    GREY  = "#95A5A6"
    WHITE = "#E0E0E0"
    ANNO  = "#BBBBBB"
    PURPLE = "#C77DFF"
    ORANGE = "#FF9F43"
 
    METHOD_COLORS = {
        "Paper": RED,
        "HAC":   TEAL,
        "SB":    GOLD,
        "Sub":   PURPLE,
        "IM":    ORANGE,
    }
    METHOD_LABELS = {
        "Paper": "Paper (Prop 4.7)",
        "HAC":   "HAC E1",
        "SB":    "Stationary Bootstrap",
        "Sub":   "Subsampling (PRW)",
        "IM":    "Ibragimov–Müller",
    }
 
    fig = plt.figure(figsize=(20, 18))
    fig.patch.set_facecolor("#0d1117")
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.62, wspace=0.40)
 
    def setup_ax(pos, title, fontsize=8):
        ax = fig.add_subplot(pos)
        ax.set_facecolor(BG)
        ax.set_title(title, color=WHITE, fontsize=fontsize,
                     fontweight="bold", pad=8)
        ax.tick_params(colors="#888888", labelsize=6.5)
        for sp in ["right", "top"]:
            ax.spines[sp].set_visible(False)
        for sp in ["bottom", "left"]:
            ax.spines[sp].set_color("#444444")
        return ax
 
    dgp_names  = df_full["DGP"].unique().tolist()
    methods    = ["Paper", "HAC", "SB", "Sub", "IM"]
    garch_df   = df_full[df_full["DGP"] == "GARCH (correct)"]
    egarch_df  = df_full[df_full["DGP"] == "EGARCH (mispecified)"]
    stable_df  = df_full[df_full["DGP"] == "Stable-law (T5.3)"]
 
    # ── Panel [1]: Coverage — Paper vs HAC ───────────────────────────
    ax1 = setup_ax(gs[0, 0], "Coverage — Paper CI vs HAC CI (E1)\n[Original Result Preserved]")
    two_dgps = ["GARCH (correct)", "EGARCH (mispecified)"]
    two_df   = df_full[df_full["DGP"].isin(two_dgps) & df_full["Method"].isin(["Paper", "HAC"])]
    x  = np.arange(2)
    w  = 0.34
    p_cover = [two_df[(two_df["DGP"]==d) & (two_df["Method"]=="Paper")]["Cover%"].values[0] for d in two_dgps]
    h_cover = [two_df[(two_df["DGP"]==d) & (two_df["Method"]=="HAC")]["Cover%"].values[0] for d in two_dgps]
    ax1.bar(x - w/2, p_cover, w, color=RED,  alpha=0.85, label="Paper (Prop 4.7)")
    ax1.bar(x + w/2, h_cover, w, color=TEAL, alpha=0.85, label="HAC E1")
    ax1.axhline(95, color=GOLD, lw=1.3, ls="--", label="95% nominal")
    ax1.set_xticks(x)
    ax1.set_xticklabels(["GARCH\n(correct)", "EGARCH\n(mispecified)"], fontsize=6.5)
    ax1.set_ylim(85, 100)
    ax1.set_ylabel("Coverage (%)", color=ANNO, fontsize=7)
    ax1.legend(fontsize=5.5, facecolor="#2d2d2d", labelcolor="white")
    for xi, v in zip(x - w/2, p_cover):
        ax1.text(xi, v + 0.15, f"{v:.1f}", ha="center", fontsize=6, color=WHITE)
    for xi, v in zip(x + w/2, h_cover):
        ax1.text(xi, v + 0.15, f"{v:.1f}", ha="center", fontsize=6, color=WHITE)
 
    # ── Panel [2]: Width — all 5 methods, GARCH ──────────────────────
    ax2 = setup_ax(gs[0, 1], "CI Width — All 5 Methods\n(GARCH correct DGP)")
    g_width = [garch_df[garch_df["Method"]==m]["Width"].values[0] for m in methods]
    x5  = np.arange(5)
    colors5 = [METHOD_COLORS[m] for m in methods]
    ax2.bar(x5, g_width, color=colors5, alpha=0.85)
    ax2.set_xticks(x5)
    ax2.set_xticklabels(["Paper", "HAC", "SB", "Sub", "IM"], fontsize=7)
    ax2.set_ylabel("Mean CI Width", color=ANNO, fontsize=7)
    for xi, v in zip(x5, g_width):
        if np.isfinite(v):
            ax2.text(xi, v + 0.0005, f"{v:.4f}", ha="center", fontsize=5.5, color=WHITE)
 
    # ── Panel [3]: V_sym vs φ ─────────────────────────────────────────
    ax3 = setup_ax(gs[0, 2],
                   "Paper Prop 4.7: V_sym grows as φ → 1\n(4th-moment boundary)")
    mask = np.isfinite(V_curve) & (V_curve < 15)
    ax3.plot(phi_grid[mask], V_curve[mask], color=RED, lw=2.0, label="V_sym (Prop 4.7)")
    ax3.axhline(V_IID,   color=GREY,  lw=1.5, ls="--", label="V IID Gaussian")
    ax3.axhline(V_PAPER, color=TEAL,  lw=1.2, ls=":",  label=f"V at φ={PHI}")
    ax3.axvline(PHI,     color=GOLD,  lw=1.0, ls=":",  alpha=0.8)
    ax3.text(PHI + 0.004, 9.0, f"φ={PHI}", color=GOLD, fontsize=6.5)
    ax3.fill_between(phi_grid,
                     [V_IID]*len(phi_grid),
                     [v if (np.isfinite(v) and v < 15) else V_IID for v in V_curve],
                     alpha=0.10, color=RED)
    ax3.set_xlabel("φ = α + β", color=ANNO, fontsize=7)
    ax3.set_ylabel("Asymptotic Variance V", color=ANNO, fontsize=7)
    ax3.set_ylim(0.9, 12)
    ax3.legend(fontsize=5.5, facecolor="#2d2d2d", labelcolor="white")
 
    # ── Panels [4–6]: Coverage — all 5 methods × 3 DGPs ─────────────
    row2_axes = [setup_ax(gs[1, i], t) for i, t in enumerate([
        "All Methods — GARCH Regime\n(Finite 4th Moment: VALID for all)",
        "All Methods — EGARCH Regime\n(Misspecified: Paper/HAC struggle)",
        "All Methods — Stable-Law Regime\n(Theorem 5.3: HAC/Paper INVALID)",
    ])]
    row2_data = [garch_df, egarch_df, stable_df]
 
    for ax, ddf in zip(row2_axes, row2_data):
        covers = [ddf[ddf["Method"]==m]["Cover%"].values[0] for m in methods]
        ax.bar(x5, covers, color=colors5, alpha=0.85)
        ax.axhline(95, color=WHITE, lw=1.0, ls="--", alpha=0.5, label="95% nominal")
        ax.axhline(90, color=ORANGE, lw=0.8, ls=":", alpha=0.5, label="90% threshold")
        ax.set_xticks(x5)
        ax.set_xticklabels(["Paper", "HAC", "SB", "Sub", "IM"], fontsize=6.5)
        ax.set_ylim(70, 102)
        ax.set_ylabel("Coverage (%)", color=ANNO, fontsize=7)
        for xi, v in zip(x5, covers):
            color_txt = WHITE if v >= 90 else RED
            ax.text(xi, v + 0.5, f"{v:.1f}", ha="center", fontsize=6,
                    color=color_txt, fontweight="bold")
        ax.legend(fontsize=5, facecolor="#2d2d2d", labelcolor="white")
 
    # ── Panel [7]: Coverage vs T in stable regime ────────────────────
    ax7 = setup_ax(gs[2, 0],
                   "Coverage vs T — Stable-Law Regime\n(SB stalls; Sub/IM converge)")
    line_methods = ["HAC", "SB", "Sub", "IM"]
    for m in line_methods:
        sub = df_stable[df_stable["Method"] == m].sort_values("T")
        ax7.plot(sub["T"], sub["Cover%"],
                 color=METHOD_COLORS[m], lw=2.0, marker="o", ms=4,
                 label=METHOD_LABELS[m])
    ax7.axhline(95, color=WHITE, lw=1.0, ls="--", alpha=0.5)
    ax7.axhline(90, color=ORANGE, lw=0.8, ls=":", alpha=0.4)
    ax7.set_xlabel("Sample Size T", color=ANNO, fontsize=7)
    ax7.set_ylabel("Coverage (%)", color=ANNO, fontsize=7)
    ax7.set_ylim(65, 102)
    ax7.legend(fontsize=5.5, facecolor="#2d2d2d", labelcolor="white", loc="lower right")
    sb_500_row = df_stable[(df_stable["Method"]=="SB") & (df_stable["T"]==500)]
    if len(sb_500_row) > 0:
        v = sb_500_row["Cover%"].values[0]
        ax7.annotate(f"SB stalls ~{v:.0f}%\n(variance-assumption failure)",
                     xy=(500, v), xytext=(600, v-8),
                     color=GOLD, fontsize=5.5,
                     arrowprops=dict(arrowstyle="->", color=GOLD, lw=0.8))
 
    # ── Panel [8]: Width ratios across DGPs ──────────────────────────
    ax8 = setup_ax(gs[2, 1],
                   "Width Ratio (Method / Paper)\nAll Methods × 3 DGPs")
    dgp_labels = ["GARCH", "EGARCH", "Stable"]
    dgp_keys   = ["GARCH (correct)", "EGARCH (mispecified)", "Stable-law (T5.3)"]
    non_paper  = ["HAC", "SB", "Sub", "IM"]
    x_dgp      = np.arange(3)
    bar_w      = 0.18
    offsets    = np.linspace(-0.27, 0.27, 4)
    for i, m in enumerate(non_paper):
        ratios = []
        for dk in dgp_keys:
            row = df_full[(df_full["DGP"]==dk) & (df_full["Method"]==m)]
            ratios.append(row["W_ratio"].values[0] if len(row) else np.nan)
        ax8.bar(x_dgp + offsets[i], ratios, bar_w,
                color=METHOD_COLORS[m], alpha=0.85, label=METHOD_LABELS[m])
    ax8.axhline(1.0, color=WHITE, lw=1.0, ls="--", alpha=0.5, label="Ratio = 1.0")
    ax8.set_xticks(x_dgp)
    ax8.set_xticklabels(dgp_labels, fontsize=7)
    ax8.set_ylabel("Width / Paper Width", color=ANNO, fontsize=7)
    ax8.legend(fontsize=5, facecolor="#2d2d2d", labelcolor="white")
 
    # ── Panel [9]: Regime separation summary (text) ──────────────────
    ax9 = setup_ax(gs[2, 2],
                   "Regime Separation: When to Use Each Method")
    ax9.axis("off")
 
    summary_lines = [
        ("FINITE 4TH-MOMENT REGIME  (DIAG > 0)", TEAL, True),
        ("  GARCH or EGARCH returns:", WHITE, False),
        ("    Paper CI:       ✓ valid,   use directly", TEAL, False),
        ("    HAC CI (E1):    ✓ narrower, preferred", TEAL, False),
        ("    Subsampling:    ✓ valid but wider", GREY, False),
        ("    IM Group:       ✓ valid but widest", GREY, False),
        ("    Stat. Bootstrap:✓ valid, moderate", GOLD, False),
        ("", WHITE, False),
        ("STABLE-LAW REGIME  (DIAG ≤ 0, Thm 5.3)", RED, True),
        ("  HFR / crypto / extreme tails:", WHITE, False),
        ("    Paper CI:       ✗ wrong regime", RED, False),
        ("    HAC CI (E1):    ✗ LRV logic breaks", RED, False),
        ("    Stat. Bootstrap:✗ stalls ~92%", ORANGE, False),
        ("    Subsampling:    ✓ primary recommendation", PURPLE, False),
        ("    IM Group:       ✓ robustness benchmark", ORANGE, False),
        ("", WHITE, False),
        ("Extensions E1–E4 by Nihar Mahesh Jani.", GOLD, True),
        ("  Inspired by López de Prado, Porcu,", ANNO, False),
        ("  Zoonekynd & Engle (SSRN 6568702, 2026).", ANNO, False),
        ("  This is NOT the official paper code.", ANNO, False),
    ]
 
    y_pos = 0.97
    for line, color, bold in summary_lines:
        weight = "bold" if bold else "normal"
        ax9.text(0.02, y_pos, line, transform=ax9.transAxes,
                 color=color, fontsize=6.0, fontweight=weight,
                 fontfamily="monospace", va="top")
        y_pos -= 0.048
 
    fig.suptitle(
        "Robust SNR Inference under Volatility Clustering and Non-Gaussian Heavy Tails: "
        "Extensions via Subsampling and Self-Normalization\n"
        "Author: Nihar Mahesh Jani  |  niharmaheshjani@gmail.com  |  "
        "Inspired by López de Prado, Porcu, Zoonekynd & Engle (SSRN 6568702, 2026)",
        color=WHITE, fontsize=9, fontweight="bold", y=1.002
    )
 
    plt.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"✓ Extended figure saved: {out_path}")

# SECTION 12 — EMPIRICAL APPLICATION: REAL ASSET DATA

In [105]:
"""
Applies all five SNR/Sharpe-ratio CI methods from Sections 3–7 to
real daily log-returns downloaded via yfinance.
 
Assets covered (one representative per asset class):
  SPY   — S&P 500 ETF            (large-cap US equity, GARCH-like)
  QQQ   — Nasdaq-100 ETF         (tech-heavy, heavier tails)
  GLD   — SPDR Gold Shares       (commodity, low autocorrelation)
  TLT   — 20+ Year Treasury ETF  (duration, macro-sensitive)
  BTC-USD — Bitcoin               (crypto, extreme tails → stable-law regime)
  EEM   — Emerging Markets ETF   (EM risk, skewed)
 
For each asset the script:
  1. Downloads 5 years of daily adjusted close prices (2019-01-01 → 2023-12-31).
  2. Computes daily log-returns (ln P_t − ln P_{t−1}).
  3. Runs a rolling GARCH(1,1) fit to obtain the fourth-moment
     diagnostic DIAG — regime flag: DIAG > 0 = finite-fourth-moment,
     DIAG ≤ 0 = stable-law.
  4. Constructs 95% CIs with all five methods (Paper, HAC, SB, Sub, IM).
  5. Computes the annualised Sharpe ratio and annualised CI bounds.
  6. Saves results to  empirical_results.json
  7. Produces a 3×2 panel figure  empirical_figure.png
 
Designed to run inside the same Jupyter session as Sections 1–11
(all helper functions are already in scope). If run as a standalone
script, the required imports and function definitions are reproduced
at the bottom of this file.
"""

'\nApplies all five SNR/Sharpe-ratio CI methods from Sections 3–7 to\nreal daily log-returns downloaded via yfinance.\n\nAssets covered (one representative per asset class):\n  SPY   — S&P 500 ETF            (large-cap US equity, GARCH-like)\n  QQQ   — Nasdaq-100 ETF         (tech-heavy, heavier tails)\n  GLD   — SPDR Gold Shares       (commodity, low autocorrelation)\n  TLT   — 20+ Year Treasury ETF  (duration, macro-sensitive)\n  BTC-USD — Bitcoin               (crypto, extreme tails → stable-law regime)\n  EEM   — Emerging Markets ETF   (EM risk, skewed)\n\nFor each asset the script:\n  1. Downloads 5 years of daily adjusted close prices (2019-01-01 → 2023-12-31).\n  2. Computes daily log-returns (ln P_t − ln P_{t−1}).\n  3. Runs a rolling GARCH(1,1) fit to obtain the fourth-moment\n     diagnostic DIAG — regime flag: DIAG > 0 = finite-fourth-moment,\n     DIAG ≤ 0 = stable-law.\n  4. Constructs 95% CIs with all five methods (Paper, HAC, SB, Sub, IM).\n  5. Computes the annualised S

In [106]:
TRADING_DAYS = 252

In [107]:
ASSETS = {
    "SPY":     "S&P 500 (SPY)",
    "QQQ":     "Nasdaq-100 (QQQ)",
    "GLD":     "Gold (GLD)",
    "TLT":     "Long Treasury (TLT)",
    "BTC-USD": "Bitcoin (BTC-USD)",
    "EEM":     "Emerging Mkts (EEM)",
}

In [108]:
START_DATE = "2019-01-01"
END_DATE   = "2023-12-31"

In [109]:
# 12.1  DATA ACQUISITION
def fetch_log_returns(tickers, start, end):
    """
    Download adjusted close prices via yfinance and convert to
    daily log-returns.  Returns a dict {ticker: np.ndarray}.
 
    Log-returns are preferred over simple returns for the SNR analysis
    because they are approximately additive over time and their
    distribution is closer to symmetric, which is the conditional
    symmetry assumption underlying Proposition 4.7 of the paper.
    """
    print(f"\n── Downloading price data ({start} → {end}) ──")
    raw = yf.download(
        list(tickers.keys()),
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        threads=True,
    )
 
    # yfinance returns a MultiIndex when multiple tickers are requested.
    # The "Close" column (post auto_adjust) holds adjusted prices.
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw["Close"]
    else:
        prices = raw[["Close"]]
        prices.columns = list(tickers.keys())[:1]
 
    returns = {}
    for ticker in tickers:
        if ticker not in prices.columns:
            print(f"  ✗ {ticker}: not found in downloaded data — skipping")
            continue
        col = prices[ticker].dropna()
        if len(col) < 200:
            print(f"  ✗ {ticker}: insufficient data ({len(col)} rows) — skipping")
            continue
        lr = np.log(col / col.shift(1)).dropna().values
        # Remove any non-finite values (e.g. trading halts)
        lr = lr[np.isfinite(lr)]
        returns[ticker] = lr
        print(f"  ✓ {ticker:>8s}: {len(lr)} daily log-returns")
 
    return returns

In [110]:
def _plug_in_sharpe(R):
    sigma = np.std(R, ddof=1)
    return np.mean(R) / max(sigma, 1e-12)

In [111]:
def _paper_ci_from_params(sr_hat, T, alpha_param, beta_param, kz_param,
                           mu_param, s2_param, alpha_level=0.05):
    """
    Paper CI (Proposition 4.7) using the GARCH-estimated parameters.
    Falls back to None when DIAG ≤ 0 (stable-law regime).
    """
    diag = 1.0 - alpha_param**2 * kz_param - 2.0*alpha_param*beta_param - beta_param**2
    if diag <= 0 or not np.isfinite(kz_param):
        return None, None
    phi = alpha_param + beta_param
    V = (1.0
         + (mu_param**2 / (4.0 * s2_param))
         * ((1.0 - beta_param)**2 * (kz_param - 1.0) * (1.0 + phi))
         / ((1.0 - phi) * diag))
    z  = stats.norm.ppf(1.0 - alpha_level / 2.0)
    se = np.sqrt(max(V, 1e-12) / T)
    return sr_hat - z * se, sr_hat + z * se

In [112]:
def _newey_west_lrcov_local(a, b, max_lag=None):
    n = len(a)
    if max_lag is None:
        max_lag = max(int(4.0 * (n / 100.0) ** (2.0 / 9.0)), 3)
    ac = a - a.mean()
    bc = b - b.mean()
    cov = np.mean(ac * bc)
    lags = np.arange(1, min(max_lag + 1, n))
    weights = 1.0 - lags / (max_lag + 1.0)
    for h, w in zip(lags, weights):
        cov += 2.0 * w * np.dot(ac[h:], bc[:-h]) / (n - h)
    return float(cov)

In [113]:
def _hac_ci_local(R, alpha_level=0.05):
    n   = len(R)
    sr  = _plug_in_sharpe(R)
    mu  = np.mean(R)
    m2  = np.mean(R**2)
    u1  = R - mu
    u2  = R**2 - m2
    O11 = _newey_west_lrcov_local(u1, u1)
    O12 = _newey_west_lrcov_local(u1, u2)
    O22 = _newey_west_lrcov_local(u2, u2)
    s2  = np.var(R, ddof=1)
    s3  = s2 ** 1.5
    g1  = m2 / s3
    g2  = -mu / (2.0 * s3)
    V   = max(g1**2 * O11 + 2.0 * g1 * g2 * O12 + g2**2 * O22, 1e-12)
    z   = stats.norm.ppf(1.0 - alpha_level / 2.0)
    se  = np.sqrt(V / n)
    return sr - z * se, sr + z * se

In [114]:
def _subsampling_ci_local(R, alpha_level=0.05):
    n      = len(R)
    sr_hat = _plug_in_sharpe(R)
    b      = max(int(np.floor(n ** (1.0 / 3.0))), 5)
    from numpy.lib.stride_tricks import as_strided
    n_sub  = n - b + 1
    item   = R.strides[0]
    sub_m  = as_strided(R, shape=(n_sub, b), strides=(item, item))
    s_means = sub_m.mean(axis=1)
    s_vars  = ((sub_m - s_means[:, None])**2).sum(axis=1) / (b - 1)
    s_srs   = s_means / np.maximum(np.sqrt(s_vars), 1e-12)
    stats_  = np.sqrt(b) * (s_srs - sr_hat)
    q_lo    = np.percentile(stats_, 100 * alpha_level / 2.0)
    q_hi    = np.percentile(stats_, 100 * (1.0 - alpha_level / 2.0))
    return sr_hat - q_hi / np.sqrt(n), sr_hat - q_lo / np.sqrt(n)

In [115]:
def _im_ci_local(R, q=8, alpha_level=0.05):
    n   = len(R)
    q   = max(min(q, n // 10), 2)
    bsz = n // q
    srs = np.array([_plug_in_sharpe(R[j*bsz:(j+1)*bsz])
                    for j in range(q) if len(R[j*bsz:(j+1)*bsz]) >= 10])
    if len(srs) < 2:
        return np.nan, np.nan
    mu_q   = np.mean(srs)
    s_q    = np.std(srs, ddof=1)
    t_crit = stats.t.ppf(1.0 - alpha_level / 2.0, df=len(srs) - 1)
    se_q   = s_q / np.sqrt(len(srs))
    return mu_q - t_crit * se_q, mu_q + t_crit * se_q

In [116]:
def _sb_ci_local(R, B=499, seed=0, alpha_level=0.05):
    """
    Lightweight stationary bootstrap (pure NumPy, no Numba dependency).
    Uses B=499 resamples for speed on real data.
    """
    np.random.seed(seed)
    n      = len(R)
    sr_hat = _plug_in_sharpe(R)
    # Politis-White block length (simplified)
    x  = R - R.mean()
    M  = max(int(np.floor(np.sqrt(n))), 5)
    M  = min(M, n // 4)
    acf = np.array([np.mean(x[k:]*x[:-k]) if k > 0 else np.mean(x**2)
                    for k in range(M+1)])
    G   = sum(k * acf[k] for k in range(1, M+1))
    D   = (2.0 * sum(acf[k] for k in range(1, M+1)))**2
    if D < 1e-15 or G < 0:
        b_bar = max(int(n**0.25), 2)
    else:
        b_bar = int(np.clip(round((2.0*G**2/max(D,1e-15))**(1/3)*n**(1/3)), 2, n//3))
    p      = 1.0 / max(b_bar, 1)
    R_circ = np.concatenate([R, R])
    boot_stats = np.empty(B)
    for b in range(B):
        boot = np.empty(n)
        idx  = 0
        while idx < n:
            start = np.random.randint(0, n)
            blen  = min(np.random.geometric(p), n - idx)
            for k in range(blen):
                boot[idx + k] = R_circ[start + k]
            idx += blen
        mu_b = np.mean(boot)
        s2_b = np.var(boot, ddof=1)
        boot_stats[b] = np.sqrt(n) * (mu_b / max(np.sqrt(s2_b), 1e-12) - sr_hat)
    q_lo = np.percentile(boot_stats, 100 * alpha_level / 2.0)
    q_hi = np.percentile(boot_stats, 100 * (1.0 - alpha_level / 2.0))
    return sr_hat - q_hi / np.sqrt(n), sr_hat - q_lo / np.sqrt(n)

In [117]:
# 12.2  REGIME DETECTION VIA GARCH FIT
def garch_regime_flag(R: np.ndarray) -> dict:
    """
    Fit a GARCH(1,1) model with Student-t innovations (via arch library)
    and compute the fourth-moment diagnostic DIAG from Proposition 4.7.
 
    DIAG = 1 - α² κ_z - 2αβ - β²
 
    where κ_z is the fourth standardised moment of the innovation
    distribution (= 3 + 6/(df-4) for Student-t with df > 4 degrees
    of freedom, or ∞ when df ≤ 4).
 
    Returns a dict with:
      alpha, beta, phi, kz, diag, regime (str)
    """
    try:
        am = arch_model(R * 100, vol="GARCH", p=1, q=1, dist="t", rescale=False)
        res = am.fit(disp="off", show_warning=False)
        params = res.params
 
        alpha = float(params.get("alpha[1]", params.get("alpha", 0.10)))
        beta  = float(params.get("beta[1]",  params.get("beta",  0.85)))
        nu    = float(params.get("nu", 8.0))   # Student-t df parameter
 
        # Clip to plausible GARCH bounds
        alpha = float(np.clip(alpha, 1e-4, 0.49))
        beta  = float(np.clip(beta,  1e-4, 0.98))
        phi   = alpha + beta
 
        # Fourth standardised moment of Student-t(nu) innovations
        if nu > 4.0:
            kz = 3.0 + 6.0 / (nu - 4.0)
        else:
            kz = np.inf   # infinite fourth moment
 
        if np.isfinite(kz):
            diag = 1.0 - alpha**2 * kz - 2.0 * alpha * beta - beta**2
        else:
            diag = -1.0   # force stable-law flag
 
        regime = "finite-4th-moment" if diag > 0 else "stable-law"
 
        return {
            "alpha": round(alpha, 5),
            "beta":  round(beta,  5),
            "phi":   round(phi,   5),
            "kz":    round(kz,    3) if np.isfinite(kz) else "inf",
            "diag":  round(diag,  5),
            "df_t":  round(nu,    2),
            "regime": regime,
        }
 
    except Exception as e:
        # If fitting fails, return a safe default (finite regime)
        return {
            "alpha": 0.10, "beta": 0.85, "phi": 0.95,
            "kz": 6.0, "diag": 0.05, "df_t": 8.0,
            "regime": "finite-4th-moment (fit failed)",
        }

In [118]:
# 12.3  PER-ASSET ANALYSIS
def analyse_asset(ticker, label, R, seed=42):
    """
    Run the full CI suite on a single asset's return series.

    Returns a dict containing:
      - Descriptive statistics (mean, vol, skew, kurtosis)
      - GARCH-fitted parameters and regime flag
      - Daily SR estimate
      - Annualised SR (× √252) and annualised CI bounds for all 5 methods
      - Raw daily CI bounds for JSON storage
    """
    n   = len(R)
    mu  = float(np.mean(R))
    s2  = float(np.var(R, ddof=1))
    sr_daily = float(_plug_in_sharpe(R))
    sr_ann   = sr_daily * np.sqrt(TRADING_DAYS)

    skew = float(stats.skew(R))
    kurt = float(stats.kurtosis(R, fisher=True))   # excess kurtosis

    # Regime flag
    garch_info = garch_regime_flag(R)
    regime     = garch_info["regime"]

    # ── Build Paper CI (uses GARCH params) ─────────────────────────
    if garch_info["kz"] == "inf":
        lo_p, hi_p = None, None
    else:
        lo_p, hi_p = _paper_ci_from_params(
            sr_daily, n,
            garch_info["alpha"], garch_info["beta"],
            garch_info["kz"], mu, s2
        )

    # ── HAC CI ─────────────────────────────────────────────────────
    lo_h, hi_h = _hac_ci_local(R)

    # ── Stationary Bootstrap (E2) ───────────────────────────────────
    print(f"    SB bootstrap (B=499)...")
    lo_sb, hi_sb = _sb_ci_local(R, B=499, seed=seed)

    # ── Subsampling (E3) ────────────────────────────────────────────
    lo_su, hi_su = _subsampling_ci_local(R)

    # ── Ibragimov–Müller (E4) ──────────────────────────────────────
    lo_im, hi_im = _im_ci_local(R, q=8)

    # Annualise all bounds
    scale = np.sqrt(TRADING_DAYS)

    def ann(v):
        return round(float(v) * scale, 4) if (v is not None and np.isfinite(v)) else None

    return {
        "ticker":   ticker,
        "label":    label,
        "n_obs":    n,
        "regime":   regime,
        "garch":    garch_info,
        "stats": {
            "mean_daily":    round(mu,        7),
            "vol_daily":     round(np.sqrt(s2), 6),
            "skewness":      round(skew,      4),
            "excess_kurt":   round(kurt,      4),
            "sr_daily":      round(sr_daily,  6),
            "sr_ann":        round(sr_ann,    4),
        },
        "ci_daily": {
            "Paper": [round(lo_p, 6) if lo_p is not None else None,
                      round(hi_p, 6) if hi_p is not None else None],
            "HAC":   [round(lo_h, 6), round(hi_h, 6)],
            "SB":    [round(lo_sb, 6), round(hi_sb, 6)],
            "Sub":   [round(lo_su, 6), round(hi_su, 6)],
            "IM":    [round(lo_im, 6) if np.isfinite(lo_im) else None,
                      round(hi_im, 6) if np.isfinite(hi_im) else None],
        },
        "ci_ann": {
            "Paper": [ann(lo_p), ann(hi_p)],
            "HAC":   [ann(lo_h), ann(hi_h)],
            "SB":    [ann(lo_sb), ann(hi_sb)],
            "Sub":   [ann(lo_su), ann(hi_su)],
            "IM":    [ann(lo_im), ann(hi_im)],
        },
    }


In [119]:
def make_empirical_figure(results: list, out_path="empirical_figure.png"):
    """
    3 × 2 panel figure — one panel per asset.

    Each panel shows:
      • Annualised Sharpe ratio estimate (black circle)
      • 95% CI bars for all five methods, colour-coded
      • Vertical dashed line at SR = 0
      • Regime label and key statistics in the panel header

    Colour palette matches the extended figure in Section 11 for
    visual consistency throughout the paper.
    """
    BG     = "#161b22"
    RED    = "#FF6B6B"
    TEAL   = "#4ECDC4"
    GOLD   = "#FFD700"
    PURPLE = "#C77DFF"
    ORANGE = "#FF9F43"
    WHITE  = "#E0E0E0"
    ANNO   = "#BBBBBB"

    METHOD_COLORS = {
        "Paper": RED,
        "HAC":   TEAL,
        "SB":    GOLD,
        "Sub":   PURPLE,
        "IM":    ORANGE,
    }
    METHODS = ["Paper", "HAC", "SB", "Sub", "IM"]
    Y_POS   = {m: i * 0.18 for i, m in enumerate(METHODS)}

    fig = plt.figure(figsize=(20, 14))
    fig.patch.set_facecolor("#0d1117")
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.60, wspace=0.35)

    for idx, res in enumerate(results):
        row, col = divmod(idx, 2)
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor(BG)

        sr_ann = res["stats"]["sr_ann"]
        regime = res["regime"]
        regime_color = TEAL if "finite" in regime else RED
        regime_short = "Finite-4th-moment" if "finite" in regime else "⚠ Stable-law"

        title = (f"{res['label']}\n"
                 f"SR_ann={sr_ann:.3f} | "
                 f"Skew={res['stats']['skewness']:.2f} | "
                 f"ExKurt={res['stats']['excess_kurt']:.2f} | "
                 f"n={res['n_obs']}\n"
                 f"GARCH: α={res['garch']['alpha']:.3f}, β={res['garch']['beta']:.3f}, "
                 f"φ={res['garch']['phi']:.3f} | {regime_short}")
        ax.set_title(title, color=WHITE, fontsize=7.2, fontweight="bold", pad=6)

        ax.axvline(0,     color="#555555", lw=1.0, ls="--")
        ax.axvline(sr_ann, color=WHITE,    lw=0.8, ls=":", alpha=0.6)

        ci_ann = res["ci_ann"]
        for m in METHODS:
            lo, hi = ci_ann[m]
            y = Y_POS[m]
            c = METHOD_COLORS[m]
            if lo is not None and hi is not None:
                ax.barh(y, hi - lo, left=lo, height=0.12,
                        color=c, alpha=0.75, linewidth=0)
                ax.text(hi + 0.003, y, f"[{lo:.3f}, {hi:.3f}]",
                        va="center", ha="left", color=c, fontsize=5.5)
            else:
                ax.text(0.02, y, f"{m}: N/A (stable-law regime)",
                        va="center", ha="left", color=RED,
                        fontsize=5.5, transform=ax.get_yaxis_transform())

        # SR dot
        ax.scatter([sr_ann], [0.36], color=WHITE, s=40, zorder=5, marker="D")
        ax.text(sr_ann, 0.42, f"SR={sr_ann:.3f}",
                ha="center", va="bottom", color=WHITE, fontsize=6.0, fontweight="bold")

        ax.set_yticks(list(Y_POS.values()))
        ax.set_yticklabels(METHODS, fontsize=6.5, color=ANNO)
        ax.set_xlabel("Annualised Sharpe Ratio", color=ANNO, fontsize=6.5)
        ax.set_xlim(min(sr_ann - 0.35, -0.35), max(sr_ann + 0.55, 0.55))
        ax.set_ylim(-0.12, 0.55)

        for sp in ["right", "top"]:
            ax.spines[sp].set_visible(False)
        for sp in ["bottom", "left"]:
            ax.spines[sp].set_color("#444444")
        ax.tick_params(colors="#888888", labelsize=6.0)

        # Regime annotation
        ax.text(0.98, 0.97, regime_short,
                transform=ax.transAxes, ha="right", va="top",
                color=regime_color, fontsize=6.0, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#1e2530",
                          edgecolor=regime_color, alpha=0.9))

    # Legend strip
    handles = [plt.Rectangle((0,0),1,1, color=METHOD_COLORS[m], alpha=0.8)
               for m in METHODS]
    labels  = ["Paper (Prop 4.7)", "HAC E1", "Stationary Bootstrap E2",
               "Subsampling E3 (PRW)", "Ibragimov–Müller E4"]
    fig.legend(handles, labels,
               loc="lower center", ncol=5,
               fontsize=7, facecolor="#1e2530", labelcolor="white",
               framealpha=0.9, edgecolor="#555555",
               bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(
        "Section 12 — Empirical SNR Inference on Real Assets (2019–2023)\n"
        "95% Confidence Intervals: Paper | HAC (E1) | Stationary Bootstrap (E2) | "
        "Subsampling (E3) | Ibragimov–Müller (E4)\n"
        "Author: Nihar Mahesh Jani  |  niharmaheshjani@gmail.com",
        color=WHITE, fontsize=8.5, fontweight="bold", y=1.025
    )

    plt.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"✓ Empirical figure saved: {out_path}")

In [120]:
# 12.6  SUMMARY TABLE
def print_summary_table(results: list):
    """
    Print a compact summary table: one row per asset × method,
    showing annualised SR estimate, CI bounds, CI width, and regime flag.
    """
    METHODS = ["Paper", "HAC", "SB", "Sub", "IM"]
    rows = []
    for res in results:
        for m in METHODS:
            lo, hi = res["ci_ann"][m]
            width = round(hi - lo, 4) if (lo is not None and hi is not None) else None
            rows.append({
                "Asset":   res["ticker"],
                "Method":  m,
                "SR_ann":  res["stats"]["sr_ann"],
                "CI_lo":   lo,
                "CI_hi":   hi,
                "Width":   width,
                "Regime":  "finite" if "finite" in res["regime"] else "stable",
            })

    df = pd.DataFrame(rows)
    print("\n" + "=" * 80)
    print("SECTION 12 — EMPIRICAL RESULTS: Annualised Sharpe Ratio CIs")
    print("=" * 80)
    print(df.to_string(index=False))
    print("=" * 80)


# SECTION 13 — MAIN

In [121]:
# Running this script end-to-end reproduces every number in the paper
# "Robust SNR Inference under Volatility Clustering and Non-Gaussian
# Heavy Tails: Extensions via Subsampling and Self-Normalization"
# by Nihar Mahesh Jani.
#
# Sequence:
#   1. Warm up Numba JIT kernels (one-time compile cost).
#   2. Print the paper constants (verify against Table 1 of the paper).
#   3. Full Monte Carlo: 400 reps × 5 methods × 3 DGPs at T = 800.
#   4. Coverage vs T in the stable-law regime (1000 reps).
#   5. Generate the 9-panel figure and save results to JSON.

In [122]:
if __name__ == "__main__":
    # ── Warm up Numba JIT kernels ───────────────────────────────────
    # Numba compiles the first call — all subsequent calls hit the
    # cache. Running dummy inputs here means the timing in the real
    # experiment reflects computation, not compilation.
    _warm_z = np.random.standard_normal(400)
    _garch11_kernel(_warm_z, 10)
    _egarch_kernel(_warm_z, 10)
    _warm_R = np.random.standard_normal(100) * 0.03
    _stationary_bootstrap_stats(
        np.concatenate([_warm_R, _warm_R]),
        100, 5, 0.1, 0.0,
        np.zeros((5, 20), dtype=np.int32),
        np.ones((5, 20), dtype=np.int32)
    )

    # ── Reproduce paper constants ───────────────────────────────────
    print("\n── Paper constants (reproduced from López de Prado et al. 2026) ──")
    print(f"  True SR              = {TRUE_SR:.6f}")
    print(f"  V_paper (Prop 4.7)   = {V_PAPER:.5f}")
    print(f"  V_iid  (benchmark)   = {V_IID:.5f}")
    print(f"  Volatility clustering adds {100*(V_PAPER/V_IID - 1):.2f}% to variance")
    print(f"  4th-moment diagnostic = {DIAG:.5f}  (> 0 ✓)")
    print(f"  GARCH params: α={ALPHA}, β={BETA}, φ={PHI:.2f}")

    # ── Full experiment ─────────────────────────────────────────────
    print("\n── Running full experiment (T=800, 400 reps × 5 methods × 3 DGPs) ──")
    t0 = time.time()
    df_full = run_full_experiment(T=800, n_reps=400)
    print(f"   Completed in {time.time()-t0:.1f}s")

    print("\n" + "="*70)
    print("RESULTS TABLE: Coverage and Width by Method × DGP")
    print("="*70)
    print(df_full.to_string(index=False))

    # ── Stable-law regime: coverage vs T ───────────────────────────
    print("\n── Stable-law regime: coverage vs T (1000 reps) ──")
    t1 = time.time()
    df_stable = stable_regime_coverage_vs_T(
        T_values=[100, 200, 350, 500, 750, 1000, 1500, 2000], n_reps=1000)
    print(f"   Completed in {time.time()-t1:.1f}s")

    print("\n" + "="*70)
    print("STABLE-LAW REGIME: Coverage vs Sample Size")
    print("="*70)
    print(df_stable.pivot(index="T", columns="Method", values="Cover%").to_string())

    # ── V_sym curve ─────────────────────────────────────────────────
    phi_grid, V_curve = v_sym_curve()

    # ── Save JSON results ───────────────────────────────────────────
    results = {
        "paper_constants": {
            "V_paper":  round(V_PAPER, 5),
            "V_iid":    round(V_IID,   5),
            "true_sr":  round(TRUE_SR,  6),
            "phi":      PHI,
            "diag":     round(DIAG, 5),
        },
        "full_experiment": df_full.to_dict(orient="records"),
        "stable_regime_coverage_vs_T": df_stable.to_dict(orient="records"),
        "verdicts": {
            "E1_HAC_narrower_than_paper_GARCH": bool(
                df_full[(df_full["DGP"]=="GARCH (correct)") & (df_full["Method"]=="HAC")]["W_ratio"].values[0] < 1.0),
            "E2_SB_stalls_in_stable_regime": bool(
                df_stable[(df_stable["Method"]=="SB") & (df_stable["T"]==500)]["Cover%"].values[0] < 93.0),
            "E3_Sub_better_than_SB_in_stable": bool(
                df_stable[(df_stable["Method"]=="Sub") & (df_stable["T"]==500)]["Cover%"].values[0]
                > df_stable[(df_stable["Method"]=="SB") & (df_stable["T"]==500)]["Cover%"].values[0]),
            "E4_IM_robust_in_stable_regime": bool(
                df_stable[(df_stable["Method"]=="IM") & (df_stable["T"]==500)]["Cover%"].values[0] >= 88.0),
        }
    }

    with open("results_extended.json", "w") as f:
        json.dump(results, f, indent=2)
    print("\n✓ results_extended.json saved")

    # ── Generate figure ─────────────────────────────────────────────
    print("\n── Generating extended 9-panel figure ──")
    make_extended_figure(df_full, df_stable, phi_grid, V_curve,
                         out_path="result_image.png")

    # ── Print verdicts ──────────────────────────────────────────────
    print("\n" + "="*70)
    print("="*70)
    v = results["verdicts"]
    print(f"  E1 HAC narrower than Paper in GARCH: {v['E1_HAC_narrower_than_paper_GARCH']}")
    print(f"  E2 SB stalls at T=500 in stable:    {v['E2_SB_stalls_in_stable_regime']}")
    print(f"  E3 Subsampling beats SB in stable:  {v['E3_Sub_better_than_SB_in_stable']}")
    print(f"  E4 IM robust in stable regime:      {v['E4_IM_robust_in_stable_regime']}")
    print("="*70)

    # ── SECTION 12 — Empirical Application: Real Asset Data ────────
    print("\n" + "="*70)
    print("SECTION 12 — EMPIRICAL APPLICATION: REAL ASSET DATA")
    print("="*70)

    t_emp = time.time()

    # ── Step 1: Download data ───────────────────────────────────────
    emp_returns = fetch_log_returns(ASSETS, START_DATE, END_DATE)

    if not emp_returns:
        print("  ✗ No data downloaded — skipping Section 12.")
        print("    Check internet connection and yfinance access.")
    else:
        # ── Step 2: Analyse each asset ──────────────────────────────
        all_results = []
        for ticker, label in ASSETS.items():
            if ticker not in emp_returns:
                continue
            R_emp = emp_returns[ticker]
            print(f"\n── Analysing {label} (n={len(R_emp)}) ──")
            res = analyse_asset(ticker, label, R_emp, seed=42)
            all_results.append(res)
            print(f"   SR_daily = {res['stats']['sr_daily']:.5f}  "
                  f"SR_ann = {res['stats']['sr_ann']:.4f}")
            print(f"   Regime : {res['regime']}  "
                  f"(DIAG={res['garch']['diag']:.5f})")
            print(f"   HAC CI : [{res['ci_ann']['HAC'][0]:.4f}, "
                  f"{res['ci_ann']['HAC'][1]:.4f}]")
            print(f"   Sub CI : [{res['ci_ann']['Sub'][0]:.4f}, "
                  f"{res['ci_ann']['Sub'][1]:.4f}]")
            print(f"   IM  CI : [{res['ci_ann']['IM'][0]:.4f}, "
                  f"{res['ci_ann']['IM'][1]:.4f}]")

        # ── Step 3: Print summary table ─────────────────────────────
        print_summary_table(all_results)

        # ── Step 4: Save JSON ────────────────────────────────────────
        empirical_json = {
            "description": (
                "Section 12 — Empirical SNR Inference on Real Assets. "
                "Annualised Sharpe ratio CIs using Paper (Prop 4.7), "
                "HAC (E1), Stationary Bootstrap (E2), "
                "Subsampling (E3, PRW 1999), and "
                "Ibragimov–Müller (E4, JBES 2010) methods. "
                "Author: Nihar Mahesh Jani."
            ),
            "data_window": {"start": START_DATE, "end": END_DATE},
            "trading_days_per_year": TRADING_DAYS,
            "assets": all_results,
        }
        with open("empirical_results.json", "w") as f:
            json.dump(empirical_json, f, indent=2)
        print("\n✓ empirical_results.json saved")

        # ── Step 5: Generate figure ──────────────────────────────────
        print("\n── Generating empirical 3×2 figure ──")
        make_empirical_figure(all_results, out_path="empirical_figure.png")


── Paper constants (reproduced from López de Prado et al. 2026) ──
  True SR              = 0.016667
  V_paper (Prop 4.7)   = 1.01270
  V_iid  (benchmark)   = 1.00014
  Volatility clustering adds 1.26% to variance
  4th-moment diagnostic = 0.03190  (> 0 ✓)
  GARCH params: α=0.13, β=0.81, φ=0.94

── Running full experiment (T=800, 400 reps × 5 methods × 3 DGPs) ──
   Completed in 13.6s

RESULTS TABLE: Coverage and Width by Method × DGP
                 DGP Method  Cover%   Width  W_ratio
     GARCH (correct)  Paper    93.2 0.13947   1.0000
     GARCH (correct)    HAC    92.2 0.13770   0.9873
     GARCH (correct)     SB    92.5 0.13766   0.9871
     GARCH (correct)    Sub    97.0 0.15929   1.1421
     GARCH (correct)     IM    94.0 0.16370   1.1737
EGARCH (mispecified)  Paper    93.0 0.13947   1.0000
EGARCH (mispecified)    HAC    92.5 0.13712   0.9832
EGARCH (mispecified)     SB    91.0 0.13679   0.9808
EGARCH (mispecified)    Sub    95.8 0.16167   1.1592
EGARCH (mispecified)     IM   